In [20]:
import re
import numpy as np
from gensim.models import Word2Vec

In [21]:
with open("text_data.txt","r") as f:
    text = f.read()

print(text)

# text = """
# Artificial intelligence is transforming industries worldwide.
# Machine learning is a subset of AI that focuses on data driven models.
# Deep learning uses neural networks with multiple hidden layers.
# AI is widely used in healthcare finance and robotics.
# As data increases AI systems become more powerful.
# """

Artificial intelligence is transforming industries worldwide. Machine learning is a subset of AI that focuses on data driven models. Deep learning uses neural networks with multiple hidden layers. AI is widely used in healthcare finance and robotics. As data increases AI systems become more powerful.


In [22]:
sentences = re.split(r'[.!?]',text)
sentences = [s.strip() for s in sentences if s.strip()]
print(sentences)

['Artificial intelligence is transforming industries worldwide', 'Machine learning is a subset of AI that focuses on data driven models', 'Deep learning uses neural networks with multiple hidden layers', 'AI is widely used in healthcare finance and robotics', 'As data increases AI systems become more powerful']


In [24]:

stopwords = ['a','an','the','is','of','that','on','with','in','and','more','as']
processed_sentences = []
for sent in sentences:
    sent = sent.lower()
    sent = re.sub(r'[^a-z\s]','',sent)
    words = sent.split()
    words = [w for w in words if w not in stopwords]
    processed_sentences.append(words)

print(processed_sentences)

[['artificial', 'intelligence', 'transforming', 'industries', 'worldwide'], ['machine', 'learning', 'subset', 'ai', 'focuses', 'data', 'driven', 'models'], ['deep', 'learning', 'uses', 'neural', 'networks', 'multiple', 'hidden', 'layers'], ['ai', 'widely', 'used', 'healthcare', 'finance', 'robotics'], ['data', 'increases', 'ai', 'systems', 'become', 'powerful']]


In [26]:
#word embeddings
model = Word2Vec(
    processed_sentences,
    vector_size = 100,
    window = 5,
    min_count = 1,
    sg = 1  #sg = 1 => skipgram ; sg = 0 => cbow
)

print(model.wv["learning"])

[-8.6191949e-03  3.6731919e-03  5.1890076e-03  5.7375422e-03
  7.4708643e-03 -6.1669364e-03  1.1150313e-03  6.0490817e-03
 -2.8434456e-03 -6.1754445e-03 -4.1475848e-04 -8.3697150e-03
 -5.5930782e-03  7.1028811e-03  3.3572994e-03  7.2166198e-03
  6.8029575e-03  7.5299749e-03 -3.7840388e-03 -5.7115470e-04
  2.3527860e-03 -4.5243539e-03  8.3848834e-03 -9.8576527e-03
  6.7575141e-03  2.9136471e-03 -4.9343999e-03  4.3923026e-03
 -1.7415195e-03  6.7134830e-03  9.9648964e-03 -4.3611852e-03
 -6.0354441e-04 -5.7013575e-03  3.8545812e-03  2.7930290e-03
  6.8809455e-03  6.1057014e-03  9.5397066e-03  9.2695532e-03
  7.9048332e-03 -6.9949739e-03 -9.1529712e-03 -3.4518423e-04
 -3.0959849e-03  7.8870784e-03  5.9356322e-03 -1.5509141e-03
  1.5147054e-03  1.7887709e-03  7.8213066e-03 -9.5026633e-03
 -1.9909245e-04  3.4616715e-03 -9.3227200e-04  8.3804736e-03
  9.0108244e-03  6.5349955e-03 -7.0994714e-04  7.7088242e-03
 -8.5359002e-03  3.1980879e-03 -4.6409117e-03 -5.0880918e-03
  3.5920022e-03  5.37270

In [27]:
#sentence vectors
sentence_vectors = []
for words in processed_sentences:
    vectors = []
    for w in words:
        if w in model.wv:
            vectors.append(model.wv[w])
    
    if len(vectors) == 0:
        sentence_vectors.append(np.zeros(model.vector_size))
    else:
        sentence_vectors.append(np.mean(vectors,axis = 0))


In [30]:
#cosine similarity
n = len(sentence_vectors)
sim_matrix = np.zeros((n,n))
for i in range(n):
    for j in range(n):
        if i != j:
            v1 = sentence_vectors[i]
            v2 = sentence_vectors[j]
            dot = np.dot(v1,v2)
            norm = np.linalg.norm(v1)*np.linalg.norm(v2)
            if norm != 0:
                sim_matrix[i][j] = max(dot/norm,0)

sim_matrix

array([[0.        , 0.00691522, 0.05957477, 0.        , 0.27399299],
       [0.00691522, 0.        , 0.18069641, 0.21770319, 0.32881847],
       [0.05957477, 0.18069641, 0.        , 0.        , 0.08124798],
       [0.        , 0.21770319, 0.        , 0.        , 0.        ],
       [0.27399299, 0.32881847, 0.08124798, 0.        , 0.        ]])

In [32]:
#page rank algo
d = 0.85
iterations = 50
scores = np.ones(n)/n
row_sum = sim_matrix.sum(axis = 1)
row_sum[row_sum == 0] = 1
norm_matrix = sim_matrix/row_sum[:,None]
for _ in range(iterations):
    scores = ((1-d)/n) + d*(np.dot(norm_matrix.T,scores))

print(scores)

[0.15200581 0.31021205 0.14612597 0.10819294 0.28346324]


In [33]:
ranked = []
for i in range(n):
    ranked.append((scores[i],sentences[i]))

ranked = sorted(ranked,reverse = True)
print(ranked)

[(0.31021204780911105, 'Machine learning is a subset of AI that focuses on data driven models'), (0.2834632399944752, 'As data increases AI systems become more powerful'), (0.1520058060035497, 'Artificial intelligence is transforming industries worldwide'), (0.14612597051456494, 'Deep learning uses neural networks with multiple hidden layers'), (0.1081929356782994, 'AI is widely used in healthcare finance and robotics')]


In [34]:
top_n = 2
result = []
for i in range(top_n):
    result.append(ranked[i][1])

summary = '. '.join(result)
summary

'Machine learning is a subset of AI that focuses on data driven models. As data increases AI systems become more powerful'